In [ ]:
# Load data

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

import os
import sys

sys.path.insert(0, os.path.abspath(".."))


# Load the data
# file_path = '../data/Test 2 - 21 Substances/substances - modified.xlsx'
# file_path = '../data/Test 3 - 4 White Powers/white_powders_modified.xlsx'
file_path = "../data/Test 3 - 4 White Powers/white_powders_modified_bb20C.xlsx"
# file_path = '../data/Test 3 - 4 White Powers/demo_dataset.xlsx'
data = pd.read_excel(file_path)


# Ditch certain columns
# data = data[['Spectra', 'demo 3', 'demo 4']]
# data = data[['Spectra', 'Cocaine', 'Sucralose']]
# data = data[['Spectra', 'Sucralose', 'Nothing-Demo']]
# data = data[['Spectra', 'Heroin', 'Sodium Bicarbonate']]

# data = data.iloc[300:]

# Remove the last 11 columns (keeping the first 10)
# data = data.iloc[:, :-16]

# Verify the new dimensions
data.shape

air_trans = np.array(
    pd.read_excel("../data/Test 3 - 4 White Powers/Air transmittance.xlsx", header=None)
)
air_trans = air_trans[:, 1:]
atm_dist_ratio = 0.11  # Atomsphere distance ratio

basis_funcs = np.array(
    pd.read_excel("../data/Test 3 - 4 White Powers/Basis functions_4-20um.xlsx", header=None)
)
print(basis_funcs.shape)

# Extract the spectra and substances
spectra = data.iloc[:, 0]
substances = data.columns[1:]

# Plotting the 21 substances against the spectra
plt.figure(figsize=(8, 6))
for substance in substances:
    plt.plot(spectra, data[substance], label=substance)

plt.xlabel("Spectra")
plt.ylabel("Value")
plt.title("Spectra vs Values (Black body emit)")
plt.legend(loc="upper right", bbox_to_anchor=(1.15, 1), ncol=1, fontsize="small")
plt.grid(True)
plt.show()

In [ ]:
# PCA and explain variance


# Extracting the substance data (excluding the "Spectra" column) and transpose it
substance_data = data.iloc[:, 1:].T

# Standardizing the data
# scaler = StandardScaler()
# standardized_data = scaler.fit_transform(substance_data)
standardized_data = substance_data - substance_data.mean(axis=0)
# standardized_data = substance_data

# Performing PCA
pca = PCA()
pca_result = pca.fit_transform(standardized_data)


# Explained variance
explained_variance = pca.explained_variance_ratio_
cumulative_variance = explained_variance.cumsum()

# Scree plot
plt.figure(figsize=(4, 6))
plt.bar(
    range(1, len(explained_variance) + 1),
    explained_variance,
    alpha=0.5,
    align="center",
    label="Individual explained variance",
)
plt.step(
    range(1, len(cumulative_variance) + 1),
    cumulative_variance,
    where="mid",
    label="Cumulative explained variance",
)

# Set x-axis ticks to be integers
plt.xticks(ticks=range(1, len(explained_variance) + 1))

plt.ylabel("Explained variance ratio")
plt.xlabel("Principal components")
plt.title("Explained Variance by Principal Components")
plt.legend(loc="best")
plt.grid(True, which="both", linestyle="--", linewidth=0.5)
plt.show()

In [ ]:
# Assuming you have already performed PCA and have the eigenvectors
# Extracting the eigenvector for the first principal component (PC1)
pc1_eigenvector = pca.components_[0]

# Create the positive part (part 1) and negative part (part 2)
pc1_part1 = np.where(pc1_eigenvector > 0, pc1_eigenvector, 0)
pc1_part2 = np.where(pc1_eigenvector < 0, -pc1_eigenvector, 0)

# Create a 2x1 grid of subplots
fig, axs = plt.subplots(1, 2, figsize=(12, 4))

# Plot Part 1 (positive part)
for substance in substances:
    axs[0].plot(spectra, data[substance], label=substance)

axs[0].set_xlabel("Spectra")
axs[0].set_ylabel("Value")
axs[0].set_title("Substances with PC1 Eigenvector Part 1 (Negative, Inverted)")
axs[0].grid(True)
axs[0].legend()

# Plot the PC1 Part 1 on the second y-axis
ax2_part1 = axs[0].twinx()
ax2_part1.plot(spectra, pc1_part1, color="black", linestyle="--", linewidth=2, label="PC1 Part 1")
ax2_part1.axhline(0, color="red", linestyle=":", linewidth=1.5)
ax2_part1.set_ylabel("PC1 Part 1 Value", color="black")

# Plot Part 2 (negative part, inverted to positive)
for substance in substances:
    axs[1].plot(spectra, data[substance], label=substance)

axs[1].set_xlabel("Spectra")
axs[1].set_ylabel("Value")
axs[1].set_title("Substances with PC1 Eigenvector Part 2 (Positive)")
axs[1].grid(True)
axs[1].legend()

# Plot the PC1 Part 2 on the second y-axis
ax2_part2 = axs[1].twinx()
ax2_part2.plot(spectra, pc1_part2, color="black", linestyle="--", linewidth=2, label="PC1 Part 2")
ax2_part2.axhline(0, color="red", linestyle=":", linewidth=1.5)
ax2_part2.set_ylabel("PC1 Part 2 Value", color="black")

# Adjust the layout
plt.tight_layout()
plt.show()


norm_pc1_part1 = pc1_part1 / pc1_part1.max()
norm_pc1_part2 = pc1_part2 / pc1_part2.max()
print(norm_pc1_part1.max(), norm_pc1_part1.min())
basis_funcs_pc1 = np.asarray([norm_pc1_part1, norm_pc1_part2]).transpose()


def simulator(mat_emit_bb, spectra, air_trans, atm_dist_ratio, basis_funcs):
    tau_air = air_trans**atm_dist_ratio
    mat_emit_bb = np.array(mat_emit_bb)
    abso_spec = tau_air * mat_emit_bb * basis_funcs
    out = np.trapz(abso_spec, spectra, axis=0)

    return out


sub_sensor_outputs = []

for substance in substances:
    sub_sensor_out = simulator(
        data[[substance]], spectra, air_trans, atm_dist_ratio, basis_funcs_pc1
    )
    print(sub_sensor_out)
    sub_sensor_outputs.append(sub_sensor_out)

A_matrix_pc1 = np.column_stack(sub_sensor_outputs)


import pandas as pd

# Create a DataFrame with norm_pc1_part1 and norm_pc1_part2
df_pc1_parts = pd.DataFrame({"norm_pc1_part1": norm_pc1_part1, "norm_pc1_part2": norm_pc1_part2})

# Save the DataFrame to an Excel file
df_pc1_parts.to_excel("pc1_parts.xlsx", index=False)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.ndimage import gaussian_filter1d

# Assuming you have already split PC1 into Part 1 and Part 2

# Smooth the parts using Gaussian filtering
sigma = 3  # The larger the sigma, the smoother the curve
pc1_part1_smooth = gaussian_filter1d(pc1_part1, sigma=sigma)
pc1_part2_smooth = gaussian_filter1d(pc1_part2, sigma=sigma)

# Create a 2x1 grid of subplots to plot the smoothed functions
fig, axs = plt.subplots(1, 2, figsize=(12, 4))

# Plot the smoothed Part 1 (positive part)
for substance in substances:
    axs[0].plot(spectra, data[substance], label=substance)

axs[0].set_xlabel("Spectra")
axs[0].set_ylabel("Value")
axs[0].set_title("Substances with Smoothed PC1 Eigenvector Part 1 (Negative, Inverted)")
axs[0].grid(True)
axs[0].legend()

# Plot the smoothed PC1 Part 1 on the second y-axis
ax2_part1 = axs[0].twinx()
ax2_part1.plot(
    spectra,
    pc1_part1_smooth,
    color="black",
    linestyle="--",
    linewidth=2,
    label="Smoothed PC1 Part 1",
)
ax2_part1.axhline(0, color="red", linestyle=":", linewidth=1.5)
ax2_part1.set_ylabel("Smoothed PC1 Part 1 Value", color="black")

# Plot the smoothed Part 2 (negative part, inverted to positive)
for substance in substances:
    axs[1].plot(spectra, data[substance], label=substance)

axs[1].set_xlabel("Spectra")
axs[1].set_ylabel("Value")
axs[1].set_title("Substances with Smoothed PC1 Eigenvector Part 2 (Positive)")
axs[1].grid(True)
axs[1].legend()

# Plot the smoothed PC1 Part 2 on the second y-axis
ax2_part2 = axs[1].twinx()
ax2_part2.plot(
    spectra,
    pc1_part2_smooth,
    color="black",
    linestyle="--",
    linewidth=2,
    label="Smoothed PC1 Part 2",
)
ax2_part2.axhline(0, color="red", linestyle=":", linewidth=1.5)
ax2_part2.set_ylabel("Smoothed PC1 Part 2 Value", color="black")

# Adjust the layout
plt.tight_layout()
plt.show()


pc1_part1_smooth = pc1_part1_smooth / pc1_part1_smooth.max()
pc1_part2_smooth = pc1_part2_smooth / pc1_part2_smooth.max()
basis_funcs_pc1_smooth = np.asarray([pc1_part1_smooth, pc1_part2_smooth]).transpose()


def simulator(mat_emit_bb, spectra, air_trans, atm_dist_ratio, basis_funcs):
    tau_air = air_trans**atm_dist_ratio
    mat_emit_bb = np.array(mat_emit_bb)
    abso_spec = tau_air * mat_emit_bb * basis_funcs
    out = np.trapz(abso_spec, spectra, axis=0)

    return out


sub_sensor_outputs = []

for substance in substances:
    sub_sensor_out = simulator(
        data[[substance]], spectra, air_trans, atm_dist_ratio, basis_funcs_pc1_smooth
    )
    print(sub_sensor_out)
    sub_sensor_outputs.append(sub_sensor_out)

A_matrix_pc1_smooth = np.column_stack(sub_sensor_outputs)


import pandas as pd

# Create a DataFrame with norm_pc1_part1 and norm_pc1_part2
df_pc1_parts = pd.DataFrame(
    {"norm_pc1_part1_smooth": pc1_part1_smooth, "norm_pc1_part2_smooth": pc1_part2_smooth}
)

# Save the DataFrame to an Excel file
df_pc1_parts.to_excel("pc1_parts_smooth.xlsx", index=False)

In [ ]:
# !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
# This is PC2, not PC1, code is not modified.
# !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!


# Assuming you have already performed PCA and have the eigenvectors
# Extracting the eigenvector for the first principal component (PC1)
pc1_eigenvector = pca.components_[1]

# Create the positive part (part 1) and negative part (part 2)
pc1_part1 = np.where(pc1_eigenvector > 0, pc1_eigenvector, 0)
pc1_part2 = np.where(pc1_eigenvector < 0, -pc1_eigenvector, 0)

# Create a 2x1 grid of subplots
fig, axs = plt.subplots(1, 2, figsize=(12, 4))

# Plot Part 1 (positive part)
for substance in substances:
    axs[0].plot(spectra, data[substance], label=substance)

axs[0].set_xlabel("Spectra")
axs[0].set_ylabel("Value")
axs[0].set_title("Substances with PC1 Eigenvector Part 1 (Negative, Inverted)")
axs[0].grid(True)
axs[0].legend()

# Plot the PC1 Part 1 on the second y-axis
ax2_part1 = axs[0].twinx()
ax2_part1.plot(spectra, pc1_part1, color="black", linestyle="--", linewidth=2, label="PC1 Part 1")
ax2_part1.axhline(0, color="red", linestyle=":", linewidth=1.5)
ax2_part1.set_ylabel("PC1 Part 1 Value", color="black")

# Plot Part 2 (negative part, inverted to positive)
for substance in substances:
    axs[1].plot(spectra, data[substance], label=substance)

axs[1].set_xlabel("Spectra")
axs[1].set_ylabel("Value")
axs[1].set_title("Substances with PC1 Eigenvector Part 2 (Positive)")
axs[1].grid(True)
axs[1].legend()

# Plot the PC1 Part 2 on the second y-axis
ax2_part2 = axs[1].twinx()
ax2_part2.plot(spectra, pc1_part2, color="black", linestyle="--", linewidth=2, label="PC1 Part 2")
ax2_part2.axhline(0, color="red", linestyle=":", linewidth=1.5)
ax2_part2.set_ylabel("PC1 Part 2 Value", color="black")

# Adjust the layout
plt.tight_layout()
plt.show()


norm_pc1_part1 = pc1_part1 / pc1_part1.max()
norm_pc1_part2 = pc1_part2 / pc1_part2.max()
print(norm_pc1_part1.max(), norm_pc1_part1.min())
basis_funcs_pc1 = np.asarray([norm_pc1_part1, norm_pc1_part2]).transpose()


def simulator(mat_emit_bb, spectra, air_trans, atm_dist_ratio, basis_funcs):
    tau_air = air_trans**atm_dist_ratio
    mat_emit_bb = np.array(mat_emit_bb)
    abso_spec = tau_air * mat_emit_bb * basis_funcs
    out = np.trapz(abso_spec, spectra, axis=0)

    return out


sub_sensor_outputs = []

for substance in substances:
    sub_sensor_out = simulator(
        data[[substance]], spectra, air_trans, atm_dist_ratio, basis_funcs_pc1
    )
    print(sub_sensor_out)
    sub_sensor_outputs.append(sub_sensor_out)

A_matrix_pc1 = np.column_stack(sub_sensor_outputs)


import pandas as pd

# Create a DataFrame with norm_pc1_part1 and norm_pc1_part2
df_pc1_parts = pd.DataFrame({"norm_pc1_part1": norm_pc1_part1, "norm_pc1_part2": norm_pc1_part2})

# Save the DataFrame to an Excel file
df_pc1_parts.to_excel("pc2_parts.xlsx", index=False)

In [ ]:
# !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
# This is PC2, not PC1, code is not modified.
# !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

import matplotlib.pyplot as plt
import numpy as np
from scipy.ndimage import gaussian_filter1d

# Assuming you have already split PC1 into Part 1 and Part 2

# Smooth the parts using Gaussian filtering
sigma = 3  # The larger the sigma, the smoother the curve
pc1_part1_smooth = gaussian_filter1d(pc1_part1, sigma=sigma)
pc1_part2_smooth = gaussian_filter1d(pc1_part2, sigma=sigma)

# Create a 2x1 grid of subplots to plot the smoothed functions
fig, axs = plt.subplots(1, 2, figsize=(12, 4))

# Plot the smoothed Part 1 (positive part)
for substance in substances:
    axs[0].plot(spectra, data[substance], label=substance)

axs[0].set_xlabel("Spectra")
axs[0].set_ylabel("Value")
axs[0].set_title("Substances with Smoothed PC1 Eigenvector Part 1 (Negative, Inverted)")
axs[0].grid(True)
axs[0].legend()

# Plot the smoothed PC1 Part 1 on the second y-axis
ax2_part1 = axs[0].twinx()
ax2_part1.plot(
    spectra,
    pc1_part1_smooth,
    color="black",
    linestyle="--",
    linewidth=2,
    label="Smoothed PC1 Part 1",
)
ax2_part1.axhline(0, color="red", linestyle=":", linewidth=1.5)
ax2_part1.set_ylabel("Smoothed PC1 Part 1 Value", color="black")

# Plot the smoothed Part 2 (negative part, inverted to positive)
for substance in substances:
    axs[1].plot(spectra, data[substance], label=substance)

axs[1].set_xlabel("Spectra")
axs[1].set_ylabel("Value")
axs[1].set_title("Substances with Smoothed PC1 Eigenvector Part 2 (Positive)")
axs[1].grid(True)
axs[1].legend()

# Plot the smoothed PC1 Part 2 on the second y-axis
ax2_part2 = axs[1].twinx()
ax2_part2.plot(
    spectra,
    pc1_part2_smooth,
    color="black",
    linestyle="--",
    linewidth=2,
    label="Smoothed PC1 Part 2",
)
ax2_part2.axhline(0, color="red", linestyle=":", linewidth=1.5)
ax2_part2.set_ylabel("Smoothed PC1 Part 2 Value", color="black")

# Adjust the layout
plt.tight_layout()
plt.show()


pc1_part1_smooth = pc1_part1_smooth / pc1_part1_smooth.max()
pc1_part2_smooth = pc1_part2_smooth / pc1_part2_smooth.max()
basis_funcs_pc1_smooth = np.asarray([pc1_part1_smooth, pc1_part2_smooth]).transpose()


def simulator(mat_emit_bb, spectra, air_trans, atm_dist_ratio, basis_funcs):
    tau_air = air_trans**atm_dist_ratio
    mat_emit_bb = np.array(mat_emit_bb)
    abso_spec = tau_air * mat_emit_bb * basis_funcs
    out = np.trapz(abso_spec, spectra, axis=0)

    return out


sub_sensor_outputs = []

for substance in substances:
    sub_sensor_out = simulator(
        data[[substance]], spectra, air_trans, atm_dist_ratio, basis_funcs_pc1_smooth
    )
    print(sub_sensor_out)
    sub_sensor_outputs.append(sub_sensor_out)

A_matrix_pc1_smooth = np.column_stack(sub_sensor_outputs)


import pandas as pd

# Create a DataFrame with norm_pc1_part1 and norm_pc1_part2
df_pc1_parts = pd.DataFrame(
    {"norm_pc1_part1_smooth": pc1_part1_smooth, "norm_pc1_part2_smooth": pc1_part2_smooth}
)

# Save the DataFrame to an Excel file
df_pc1_parts.to_excel("pc2_parts_smooth.xlsx", index=False)

In [ ]:
basis_funcs_hand_pick_1 = basis_funcs[:, [2, 7]]


# Create a 2x1 grid of subplots to plot the smoothed functions
fig, axs = plt.subplots(1, 2, figsize=(12, 4))

# Plot the smoothed Part 1 (positive part)
for substance in substances:
    axs[0].plot(spectra, data[substance], label=substance)

axs[0].set_xlabel("Spectra")
axs[0].set_ylabel("Value")
axs[0].set_title("Substances with Smoothed PC1 Eigenvector Part 1 (Negative, Inverted)")
axs[0].grid(True)
axs[0].legend()

# Plot the smoothed PC1 Part 1 on the second y-axis
ax2_part1 = axs[0].twinx()
ax2_part1.plot(
    spectra,
    basis_funcs_hand_pick_1[:, 0],
    color="black",
    linestyle="--",
    linewidth=2,
    label="Smoothed PC1 Part 1",
)
ax2_part1.axhline(0, color="red", linestyle=":", linewidth=1.5)
ax2_part1.set_ylabel("Smoothed PC1 Part 1 Value", color="black")

# Plot the smoothed Part 2 (negative part, inverted to positive)
for substance in substances:
    axs[1].plot(spectra, data[substance], label=substance)

axs[1].set_xlabel("Spectra")
axs[1].set_ylabel("Value")
axs[1].set_title("Substances with Smoothed PC1 Eigenvector Part 2 (Positive)")
axs[1].grid(True)
axs[1].legend()

# Plot the smoothed PC1 Part 2 on the second y-axis
ax2_part2 = axs[1].twinx()
ax2_part2.plot(
    spectra,
    basis_funcs_hand_pick_1[:, 1],
    color="black",
    linestyle="--",
    linewidth=2,
    label="Smoothed PC1 Part 2",
)
ax2_part2.axhline(0, color="red", linestyle=":", linewidth=1.5)
ax2_part2.set_ylabel("Smoothed PC1 Part 2 Value", color="black")

# Adjust the layout
plt.tight_layout()
plt.show()


def simulator(mat_emit_bb, spectra, air_trans, atm_dist_ratio, basis_funcs):
    tau_air = air_trans**atm_dist_ratio
    mat_emit_bb = np.array(mat_emit_bb)
    abso_spec = tau_air * mat_emit_bb * basis_funcs
    out = np.trapz(abso_spec, spectra, axis=0)

    return out


sub_sensor_outputs = []

for substance in substances:
    sub_sensor_out = simulator(
        data[[substance]], spectra, air_trans, atm_dist_ratio, basis_funcs_hand_pick_1
    )
    print(sub_sensor_out)
    sub_sensor_outputs.append(sub_sensor_out)

A_matrix_hand_pick_1 = np.column_stack(sub_sensor_outputs)

In [ ]:
basis_funcs_hand_pick_2 = basis_funcs[:, [4, 5]]


# Create a 2x1 grid of subplots to plot the smoothed functions
fig, axs = plt.subplots(1, 2, figsize=(12, 4))

# Plot the smoothed Part 1 (positive part)
for substance in substances:
    axs[0].plot(spectra, data[substance], label=substance)

axs[0].set_xlabel("Spectra")
axs[0].set_ylabel("Value")
axs[0].set_title("Substances with Smoothed PC1 Eigenvector Part 1 (Negative, Inverted)")
axs[0].grid(True)
axs[0].legend()

# Plot the smoothed PC1 Part 1 on the second y-axis
ax2_part1 = axs[0].twinx()
ax2_part1.plot(
    spectra,
    basis_funcs_hand_pick_2[:, 0],
    color="black",
    linestyle="--",
    linewidth=2,
    label="Smoothed PC1 Part 1",
)
ax2_part1.axhline(0, color="red", linestyle=":", linewidth=1.5)
ax2_part1.set_ylabel("Smoothed PC1 Part 1 Value", color="black")

# Plot the smoothed Part 2 (negative part, inverted to positive)
for substance in substances:
    axs[1].plot(spectra, data[substance], label=substance)

axs[1].set_xlabel("Spectra")
axs[1].set_ylabel("Value")
axs[1].set_title("Substances with Smoothed PC1 Eigenvector Part 2 (Positive)")
axs[1].grid(True)
axs[1].legend()

# Plot the smoothed PC1 Part 2 on the second y-axis
ax2_part2 = axs[1].twinx()
ax2_part2.plot(
    spectra,
    basis_funcs_hand_pick_2[:, 1],
    color="black",
    linestyle="--",
    linewidth=2,
    label="Smoothed PC1 Part 2",
)
ax2_part2.axhline(0, color="red", linestyle=":", linewidth=1.5)
ax2_part2.set_ylabel("Smoothed PC1 Part 2 Value", color="black")

# Adjust the layout
plt.tight_layout()
plt.show()


def simulator(mat_emit_bb, spectra, air_trans, atm_dist_ratio, basis_funcs):
    tau_air = air_trans**atm_dist_ratio
    mat_emit_bb = np.array(mat_emit_bb)
    abso_spec = tau_air * mat_emit_bb * basis_funcs
    out = np.trapz(abso_spec, spectra, axis=0)

    return out


sub_sensor_outputs = []

for substance in substances:
    sub_sensor_out = simulator(
        data[[substance]], spectra, air_trans, atm_dist_ratio, basis_funcs_hand_pick_2
    )
    print(sub_sensor_out)
    sub_sensor_outputs.append(sub_sensor_out)

A_matrix_hand_pick_2 = np.column_stack(sub_sensor_outputs)

In [ ]:
import numpy as np


def trace(A):
    """
    Calculates the trace of the inverse of (A^T A).

    Parameters:
    A (numpy.ndarray): Input matrix A.

    Returns:
    float: The trace of the inverse of (A^T A).

    Raises:
    LinAlgError: If (A^T A) is singular and cannot be inverted.
    """
    # Compute A^T * A
    A_T_A = np.dot(A.T, A)

    # Try to compute the inverse of A^T * A
    try:
        inv_A_T_A = np.linalg.inv(A_T_A)
    except np.linalg.LinAlgError:
        # If singular, raise an error or handle accordingly
        raise np.linalg.LinAlgError("A^T * A is singular and cannot be inverted.")
        # Alternatively, you can use the pseudo-inverse:
        # inv_A_T_A = np.linalg.pinv(A_T_A)

    # Calculate the trace of the inverse matrix
    trace_value = np.trace(inv_A_T_A)
    return trace_value

In [ ]:
t_pc1 = trace(A_matrix_pc1)
t_pc1_smooth = trace(A_matrix_pc1_smooth)
t_hp1 = trace(A_matrix_hand_pick_1)
t_hp2 = trace(A_matrix_hand_pick_2)


print(t_pc1)
print(t_pc1_smooth)
print(t_hp1)
print(t_hp2)